# Plan
 - Create directory of .inp files that span node range for both linear and non-linear tetrahedrons
 - run: "abaqus job=MyJob datacheck" for each .inp file
 - check the predicted memory usage
---
 - Should also check effect of chopping mc1

In [12]:
from pathlib import Path
import pyvista as pv
import pandas as pd
import subprocess

from phd_helpers.paths import linear_to_quadratic_mesh

In [15]:
path_MeshPipeline_main = '../../../../MeshPipeline/main.py'
subprocess.run(["python", path_MeshPipeline_main])


Updating parameters.json
	Wrote /Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/WorkPackages/WorkPackages/TMCJ-Contact/Computational/MeshPipeline/set_parameters/parameters.json


Full parameter file saved to outputs/StMmLt/params/full_params.json

SUBJECT: 14874R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 8.581s - ok
		STEP: cartilage
			RUN ID: -0-0
			Runtime: 14.855s - ok
		STEP: 3Dmesh
			RUN ID: -0-0-0
			Runtime: 284.135s - ok
	BONES: mc1-tpm
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 3.121s - ok
		STEP: cartilage
			RUN ID: -0-0
			Runtime: 20.918s - ok
		STEP: 3Dmesh
			RUN ID: -0-0-0
			Runtime: 256.727s - ok

SUBJECT: 22306R
	BONES: tpm-mc1
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 8.318s - ok
		STEP: cartilage
			RUN ID: -0-0
			Runtime: 15.950s - ok
		STEP: 3Dmesh
			RUN ID: -0-0-0
			Runtime: 64.558s - ok
	BONES: mc1-tpm
		STEP: 2Dmesh
			RUN ID: -0
			Runtime: 3.279s - ok
		STEP: cartilage
			RUN ID: -0-0
			Runtime: 23.642s - ok
		STEP: 

CompletedProcess(args=['python', '../../../../MeshPipeline/main.py'], returncode=0)

### ~8x N vertices from linear to quadratic

In [ ]:
def get_sub(path):
    return path.parent.parent.parent.name
def get_bone(path):
    return path.parent.parent.name.split('-')[0]

mesh_paths = list(Path('outputs').glob('**/*.vtu'))

for mesh_path in mesh_paths:
    mesh = pv.read(mesh_path).extract_cells_by_type(10)
    quad = linear_to_quadratic_mesh(mesh)
    print('\n')
    print(get_sub(mesh_path), get_bone(mesh_path))
    print('\telements: ', mesh.n_cells, quad.n_cells)
    print('\tvertices: ', mesh.n_points, quad.n_points)



14874R mc1
	elements:  1162979 1162979
	vertices:  209359 1621161


14874R tpm
	elements:  1140785 1140785
	vertices:  205059 1588771


22306R mc1
	elements:  328818 328818
	vertices:  60028 461539


22306R tpm
	elements:  309135 309135
	vertices:  56276 433411


50037L mc1
	elements:  169928 169928
	vertices:  31525 240499


50037L tpm
	elements:  177452 177452
	vertices:  32736 250475


## Write these meshes to .inp files then check memeory estimate with Abaqus

#### Old builder

In [3]:
path_MeshPipeline_main = '../../../../InpPipeline/main.py'
subprocess.run(["python", path_MeshPipeline_main])


Updating parameters.json
	Wrote /Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/WorkPackages/WorkPackages/TMCJ-Contact/Computational/inpPipeline/set_parameters/parameters.json
Full parameter file saved to outputs/StMmLt-inps/params/full_params-13.json

SUBJECT: 22306R
	MESH: 0-0-0
		RUN ID: 0
			Runtime: 10.339s - ok
		RUN ID: 1
			Runtime: 37.017s - ok

SUBJECT: 50037L
	MESH: 0-0-0
		RUN ID: 0
			Runtime: 5.323s - ok
		RUN ID: 1
			Runtime: 20.118s - ok

SUBJECT: 14874R
	MESH: 0-0-0
		RUN ID: 0
			Runtime: 41.807s - ok
		RUN ID: 1
			Runtime: 201.541s - ok


CompletedProcess(args=['python', '../../../../InpPipeline/main.py'], returncode=0)

In [2]:
tpm = pv.read('/Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/WorkPackages/WorkPackages/TMCJ-Contact/Computational/notebooks/MeshPipeline/ParamOptimisation/AbaqusMemoryCheck.ipynb/outputs/StMmLt/meshes/14874R/tpm-mc1/3Dmesh/mesh-0-0-0.vtu')
quad = linear_to_quadratic_mesh(tpm)

mc1 = pv.read('/Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/WorkPackages/WorkPackages/TMCJ-Contact/Computational/notebooks/MeshPipeline/ParamOptimisation/AbaqusMemoryCheck.ipynb/outputs/StMmLt/meshes/14874R/mc1-tpm/3Dmesh/mesh-0-0-0.vtu')
quad_mc1 = linear_to_quadratic_mesh(mc1)

In [2]:
import re
from pathlib import Path

def parse_memory_estimate(dat_file):
    """
    Parse Abaqus .dat file MEMORY ESTIMATE table.

    Returns:
        {
            "process": 1,
            "minimum_memory_required_mb": 354,
            "memory_to_minimize_io_mb": 3380,
        }
    """
    text = Path(dat_file).read_text(errors="ignore")

    pattern = re.compile(
        r"""
        M\s*E\s*M\s*O\s*R\s*Y\s+E\s*S\s*T\s*I\s*M\s*A\s*T\s*E   # heading
        .*?                                                    # table header
        ^\s*(\d+)\s+                                           # process
        [0-9.E+-]+\s+                                          # floating point ops
        (\d+)\s+                                               # minimum memory (MB)
        (\d+)                                                  # memory to minimize I/O (MB)
        \s*$
        """,
        re.IGNORECASE | re.DOTALL | re.MULTILINE | re.VERBOSE,
    )

    m = pattern.search(text)
    if not m:
        raise ValueError("Could not find Abaqus MEMORY ESTIMATE table in .dat file")

    return {
        "process": int(m.group(1)),
        "minimum_memory_required_gb": int(m.group(2))/1e3,
        "memory_to_minimize_io_gb": int(m.group(3))/1e3,
    }

In [9]:
sub = '22306R'
pose = 'neutral'
run_id = '1'
run_id_mesh = '0-0-0'

job_name = f'{run_id_mesh}-{pose}-{run_id}'
dat_path = f'outputs/StMmLt-inps/inpFiles/{sub}/inp/{job_name}/{job_name}.dat'
data = parse_memory_estimate(dat_path)
data


{'process': 1,
 'minimum_memory_required_gb': 4.903,
 'memory_to_minimize_io_gb': 118.703}

In [10]:
subs = ['50037L', '22306R', '14874R']
pose = 'neutral'
run_ids = ['0', '1']
mems = []
for sub in subs:
    for run_id in run_ids:
        job_name = f'{run_id_mesh}-{pose}-{run_id}'
        dat_path = f'outputs/StMmLt-inps/inpFiles/{sub}/inp/{job_name}/{job_name}.dat'
        data = parse_memory_estimate(dat_path)
        data['sub'] = sub
        data['run_id'] = run_id
        mems.append(data)

In [14]:
mems_df = pd.DataFrame(mems)
mems_df[['sub', 'run_id', 'minimum_memory_required_gb', 'memory_to_minimize_io_gb']]

,sub,run_id,minimum_memory_required_gb,memory_to_minimize_io_gb
0,50037L,0,0.354,3.380
1,50037L,1,3.081,52.489
2,22306R,0,0.703,7.758
3,22306R,1,4.903,118.703
4,14874R,0,2.609,38.050
5,14874R,1,13.235,575.808


In [ ]:
HAVE TO TRY 1 LAYER OF 10 NODE TETRAHEDRONS
also 2 layers

This needs to be mesh independance study
     - do it with easy subject
     - Try 1 and 2 and 3 layers of both 4 and 10 node tets
        - measure: computation time, contact area, peak pressure

could test how much slower in vs out of memory is just by limiting memory to 50-75% of required for the run

use 50000R for mesh independence cos expect high deformation and small contact area and high peak pressure 
        - also has potentially reasonable amount of cells.
        - 3 layers of T10 expected to be not too far outside of 64gb